In [1]:
%%capture output
%pip install -U huggingface_hub transformers
import os
import math
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from huggingface_hub import login, HfApi, notebook_login
from torchvision import datasets, transforms
from tqdm import tqdm

In [2]:
TRAIN_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/train' # Update if your path is different
TEST_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/test'
IMG_SIZE = 224
CLASSES = {
    'asian': 0, 'boho': 1, 'coastal': 2, 'contemporary': 3, 'craftsman': 4,
    'eclectic': 5, 'farmhouse': 6, 'french-country': 7, 'industrial': 8,
    'mediterranean': 9, 'minimalist': 10, 'modern': 11, 'scandinavian': 12,
    'shabby-chic-style': 13, 'southwestern': 14, 'tropical': 15, 'victorian': 16
}


In [3]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [4]:
class PatchEmbed(nn.Module):
    """Split image into patches and embed them."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        x = self.proj(x) 
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2) 
        return x, (H, W)


In [5]:
class LayerScale(nn.Module):
    """Layer Scale for DINOv3"""
    def __init__(self, dim, init_values=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim) * init_values)
    
    def forward(self, x):
        return x * self.gamma


In [6]:
def rope_rotate_half(x):
    """Rotate half the hidden dims of the input."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def rope_apply(x, sin, cos):
    """Apply rotary embedding to x."""
    return (x * cos) + (rope_rotate_half(x) * sin)


In [7]:
class RopePositionEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) for DINOv3"""
    def __init__(self, embed_dim, num_heads, base=100.0):
        super().__init__()
        assert embed_dim % (4 * num_heads) == 0
        head_dim = embed_dim // num_heads
        self.head_dim = head_dim
        self.num_heads = num_heads
        
        D_head_4 = head_dim // 4
        inv_freq = 1.0 / (base ** (torch.arange(0, D_head_4, dtype=torch.float32) / D_head_4))
        self.register_buffer('inv_freq', inv_freq)
    
    def forward(self, H, W):
        """Generate RoPE embeddings for H x W grid"""
        device = self.inv_freq.device
        dtype = self.inv_freq.dtype
        
        coords_h = torch.arange(0.5, H, device=device, dtype=dtype) / H  # [H]
        coords_w = torch.arange(0.5, W, device=device, dtype=dtype) / W  # [W]
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'), dim=-1)  # [H, W, 2]
        coords = coords.flatten(0, 1)  
        coords = 2.0 * coords - 1.0  
        
        angles = 2 * math.pi * coords[:, :, None] / self.inv_freq[None, None, :]
        angles = angles.reshape(-1, self.head_dim // 2) 
        
        cos = torch.cos(angles)  
        sin = torch.sin(angles)  
        
        cos = cos.unsqueeze(0).expand(self.num_heads, -1, -1)
        sin = sin.unsqueeze(0).expand(self.num_heads, -1, -1)
        
        cos = cos.repeat_interleave(2, dim=-1)  
        sin = sin.repeat_interleave(2, dim=-1) 
        
        return sin, cos


In [8]:
class SelfAttention(nn.Module):
    """Self Attention with RoPE support"""
    def __init__(self, dim, num_heads=8, qkv_bias=True, proj_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)
    
    def apply_rope(self, q, k, rope):
        """Apply RoPE to queries and keys"""
        if rope is None:
            return q, k
        sin, cos = rope
        q_dtype = q.dtype
        k_dtype = k.dtype
        
        B, num_heads, N, head_dim = q.shape
        
        prefix_len = N - sin.shape[1]
        if prefix_len > 0:
            q_prefix = q[:, :, :prefix_len, :]
            k_prefix = k[:, :, :prefix_len, :]
            q_patches = q[:, :, prefix_len:, :]
            k_patches = k[:, :, prefix_len:, :]
            
            q_patches = q_patches.to(sin.dtype)
            k_patches = k_patches.to(sin.dtype)
            q_patches = rope_apply(q_patches, sin.unsqueeze(0), cos.unsqueeze(0))
            k_patches = rope_apply(k_patches, sin.unsqueeze(0), cos.unsqueeze(0))
            
            q = torch.cat([q_prefix.to(q_dtype), q_patches.to(q_dtype)], dim=2)
            k = torch.cat([k_prefix.to(k_dtype), k_patches.to(k_dtype)], dim=2)
        else:
            q = rope_apply(q.to(sin.dtype), sin.unsqueeze(0), cos.unsqueeze(0)).to(q_dtype)
            k = rope_apply(k.to(sin.dtype), sin.unsqueeze(0), cos.unsqueeze(0)).to(k_dtype)
        
        return q, k
    
    def forward(self, x, rope=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        q, k = self.apply_rope(q, k, rope)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


In [9]:
class Mlp(nn.Module):
    """MLP with GELU activation"""
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0., bias=True):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        
        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias)
        self.drop2 = nn.Dropout(drop)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


In [10]:
class SelfAttentionBlock(nn.Module):
    """DINOv3 Transformer Block with LayerScale"""
    def __init__(self, dim, num_heads, ffn_ratio=4.0, qkv_bias=True, proj_bias=True, 
                 ffn_bias=True, drop=0., attn_drop=0., drop_path=0., layerscale_init=1e-5):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = SelfAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias, 
                                  proj_bias=proj_bias, attn_drop=attn_drop, proj_drop=drop)
        self.ls1 = LayerScale(dim, init_values=layerscale_init) if layerscale_init else nn.Identity()
        
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        mlp_hidden_dim = int(dim * ffn_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, 
                       act_layer=nn.GELU, drop=drop, bias=ffn_bias)
        self.ls2 = LayerScale(dim, init_values=layerscale_init) if layerscale_init else nn.Identity()
        
        self.drop_path = drop_path
    
    def forward(self, x, rope=None):
        x = x + self.ls1(self.attn(self.norm1(x), rope=rope))
        x = x + self.ls2(self.mlp(self.norm2(x)))
        return x


In [11]:
class DinoVisionTransformer(nn.Module):
    """DINOv3 Vision Transformer from scratch"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, ffn_ratio=4.0, qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0., layerscale_init=1e-5,
                 pos_embed_rope_base=100.0):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.patch_size = patch_size
        
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        self.rope_embed = RopePositionEmbedding(embed_dim, num_heads, base=pos_embed_rope_base)
        
        self.blocks = nn.ModuleList([
            SelfAttentionBlock(
                dim=embed_dim, num_heads=num_heads, ffn_ratio=ffn_ratio,
                qkv_bias=qkv_bias, proj_bias=True, ffn_bias=True,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=drop_path_rate,
                layerscale_init=layerscale_init
            )
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()
        
        self.apply(self._init_weights)
        nn.init.normal_(self.cls_token, std=0.02)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
    
    def forward_features(self, x):
        B = x.shape[0]
        x, (H, W) = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        rope = self.rope_embed(H, W)
        
        for blk in self.blocks:
            x = blk(x, rope=rope)
        
        x = self.norm(x)
        return x[:, 0] 
    
    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x


In [12]:
def get_dinov3_base(num_classes_target=17):
    """
    Load official DINOv3 ViT-Base pretrained weights and build a classifier on top.
    """
    cfg = {
        "img_size": 224,
        "patch_size": 16,
        "embed_dim": 768,
        "depth": 12,
        "num_heads": 12,
        "url": "https://dinov3.llamameta.net/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiZWh5ZDFld2ZlOWQ2aW5yaXZoaXhnZmltIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NjU3MTk5MDZ9fX1dfQ__&Signature=pWuubxC8QqlaVfmxANPRnsU-5YcTwoai3BH-BHWPcnL9fJeWBEU-6UoWcGFAzoErm2z%7E0th15iDiVcRnjUo298%7E8R7V-CLG3122mIR%7E5yiaKHbeFz28ZzqhZAg8LLekq-UGWliMVW8eK6xBHskCoPTqcCh6fVDL5u4hv5Rk5jknteU8SyMLI1AOXYKGCTpnkYgRVFTq9LeIK2cA7nu4bzxG5IZ9-CdxM%7EC-57j%7EbxxogdUkfPKtucVkbEMJ63JBJcwHnCUA3-xNCCAtraVHpWH52mZIXTkDlrLWQkWvlgPzGU0zMNQYXcj9TY0eMu3M8aU7GLhOL8LV69840Y-VCFg__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1120347550175607"
    }

    model = DinoVisionTransformer(
        img_size=cfg["img_size"],
        patch_size=cfg["patch_size"],
        embed_dim=cfg["embed_dim"],
        depth=cfg["depth"],
        num_heads=cfg["num_heads"],
        ffn_ratio=4.0,
        qkv_bias=True,
        num_classes=num_classes_target,
        layerscale_init=1e-5,
    )

    print("Downloading DINOv3 ViT-Base pretrained weights...")
    checkpoint = torch.hub.load_state_dict_from_url(
        cfg["url"], map_location="cpu", progress=True
    )

    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    else:
        state_dict = checkpoint

    clean_state_dict = {
        k.replace("module.", ""): v for k, v in state_dict.items()
    }

    missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)

    print(f"Missing keys: {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")

    nn.init.trunc_normal_(model.head.weight, std=0.02)
    if model.head.bias is not None:
        nn.init.constant_(model.head.bias, 0)

    print("DINOv3 ViT-Base ready.")
    return model


In [ ]:
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf)

targets = full_dataset.targets
class_counts = Counter(targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

full_loader = DataLoader(
    full_dataset,
    batch_size=128,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = get_dinov3_base(num_classes_target=17)

model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)
criterion = nn.CrossEntropyLoss()


In [ ]:
def train_full_data(model, loader, criterion, optimizer, scheduler, device, epochs=12):
    print(f"Starting training on {len(loader.dataset)} images for {epochs} epochs.")
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        model.train()
        
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(loader, leave=True)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_description(f"Loss: {loss.item():.4f}")
            
        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc = correct / total
        
        print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
        
        scheduler.step()

    save_path = "dinov3_base_full_data.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Training Complete. Model saved to {save_path}")
    return model


model = train_full_data(
    model, 
    full_loader, 
    criterion, 
    optimizer, 
    scheduler, 
    device, 
    epochs=8
)


In [15]:
# notebook_login()
# repo_id = "mostafafaheem/DINOv3-classification" 

# api = HfApi()

# api.create_repo(repo_id=repo_id, exist_ok=True)

# api.upload_file(
#     path_or_fileobj="/kaggle/working/dinov3_base_full_data.pth",
#     path_in_repo="model.pth",
#     repo_id=repo_id,
#     repo_type="model"
# )
# print(f"Success! Model uploaded to: https://huggingface.co/{repo_id}")

In [16]:
test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.filenames = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.root_dir, filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Warning: Corrupted image found: {filename}. Using dummy input.")
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
        
        if self.transform:
            image = self.transform(image)
            
        return image, filename


In [17]:
def load_model(num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model_config = {
        "img_size": 224,
        "patch_size": 16,
        "embed_dim": 768,
        "depth": 12,
        "num_heads": 12,
        "ffn_ratio": 4.0,
        "qkv_bias": True,
        "layerscale_init": 1e-5,
        "num_classes": num_classes
    }
    
    model = DinoVisionTransformer(**model_config)

    model_path = hf_hub_download(
        repo_id="mostafafaheem/DINOv3-classification", 
        filename="model.pth",
        repo_type="model" 
    )
    
    state_dict = torch.load(model_path, map_location="cpu")
    
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "") 
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict)
    
    model.to(device)
    model.eval()
    return model, device

model, device = load_model(17)

test_dataset = TestDataset(TEST_DIR, transform=test_tf)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

all_filenames = []
all_predictions = []

with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_filenames.extend(filenames)
        all_predictions.extend(preds.cpu().numpy())

df = pd.DataFrame({
    'ImageName': all_filenames,
    'ClassLabel': all_predictions
})
df.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
print(f"Total predictions: {len(all_predictions)}")


model.pth:   0%|          | 0.00/343M [00:00<?, ?B/s]

  0%|          | 0/43 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 26%|██▌       | 11/43 [00:23<00:55,  1.74s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 28%|██▊       | 12/43 [00:24<00:56,  1.82s/it]

 30%|███       | 13/43 [00:26<00:50,  1.68s/it]

 44%|████▍     | 19/43 [00:38<00:43,  1.82s/it]

 74%|███████▍  | 32/43 [01:04<00:22,  2.09s/it]

100%|██████████| 43/43 [01:25<00:00,  1.99s/it]

Submission saved to submission.csv
Total predictions: 5482
